# Offline synthetic reservoir tutorial
Run a four-day synthetic case and check its water balance without calling a model provider. Open this notebook from a repository clone and select the virtual environment where you installed HydroReOpLLM.

In [ ]:
from pathlib import Path
import json, tempfile, yaml

root = Path.cwd()
if not (root / 'examples' / 'daily_offline' / 'single.json').exists():
    root = root.parent
case_path = root / 'examples' / 'daily_offline' / 'single.json'
config_path = root / 'examples' / 'daily_offline' / 'single.yaml'
case = json.loads(case_path.read_text(encoding='utf-8'))
config = yaml.safe_load(config_path.read_text(encoding='utf-8'))
len(case['inflow_records']), config['agents']['operator_1']['mode'], config['max_calls']

## Expected storage
For each daily step, storage changes by `(inflow - controlled release) * 86,400`. The fixture has no spill because storage stays below capacity.

In [ ]:
physical = case['network']['reservoirs']['Lake_1']['physical']
release = config['agents']['operator_1']['release_cms']
storage = physical['initial_storage_m3']
trajectory = []
for row in case['inflow_records']:
    storage += (row['local_inflow_cms']['local_1'] - release) * physical['dt_seconds']
    trajectory.append({'date': row['date'], 'storage_m3': storage})
trajectory

## Run the real workflow
Jupyter removes the temporary output directory when this cell finishes. To keep the run files, follow [Run the offline example](../docs/GETTING_STARTED.md#run-the-offline-example) and see the [results guide](../docs/RESULTS_GUIDE.md) to read them. `--require-offline` verifies that the completed run used no provider transport.

In [ ]:
import shutil, subprocess, sys
import pandas as pd

with tempfile.TemporaryDirectory(prefix='hydroreopllm-tutorial-') as tmp:
    work = Path(tmp)
    shutil.copy(case_path, work / 'single.json')
    shutil.copy(config_path.parent / 'operator.txt', work / 'operator.txt')
    local = dict(config)
    local['run_id'] = 'notebook-offline'
    local['output'] = str(work / 'run')
    local_config = work / 'single.yaml'
    local_config.write_text(yaml.safe_dump(local, sort_keys=False), encoding='utf-8')
    command = [sys.executable, '-m', 'llm_reservoir_agent.workflow']
    def invoke(*args):
        completed = subprocess.run(command + list(args), text=True, capture_output=True)
        if completed.returncode:
            raise RuntimeError(completed.stderr or completed.stdout)
        return completed.stdout
    invoke('check', str(local_config))
    invoke('run', str(local_config))
    verified = invoke('verify', str(work / 'run'), '--require-offline')
    report = json.loads(verified)
    trajectory_df = pd.read_csv(work / 'run' / 'reservoir_trajectory.csv')
report

## Inspect the simulated trajectory
`reservoir_trajectory.csv` records the requested `target_release_cms`, the physically applied `controlled_release_cms`, `spill_cms`, and `storage_end_m3` at every step. Confirm that the engine's `storage_end_m3` matches the hand-calculated `Expected storage` above.

In [ ]:
expected = [row['storage_m3'] for row in trajectory]
actual = trajectory_df['storage_end_m3'].tolist()
assert all(abs(a - b) <= 1e-6 for a, b in zip(actual, expected, strict=True))
columns = [
    'date',
    'storage_start_m3',
    'inflow_cms',
    'target_release_cms',
    'controlled_release_cms',
    'spill_cms',
    'storage_end_m3',
]
trajectory_df[columns]

## Next steps
- Trace downstream routing across three reservoirs in `examples/daily_offline/cascade.yaml`.
- Swap the fixed release for a flood-control rule in `examples/daily_offline/single.mosart.yaml`.
- Explore a coupled HydroCNHS two-reservoir study in `examples/multiple_reservoirs/experiment.yaml`.
- Read [Results guide](../docs/RESULTS_GUIDE.md) for the full list of saved CSV and JSONL files.